# Week 1 — Run it, then discover a real truth yourself

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rahadul231/flyrank-ml-internship-starter/blob/main/notebooks/01_first_look_and_discovery.ipynb?flush_cache=true)

By the end of this notebook you will have:
1. **Run a real ML pipeline** on real (anonymized) search data and watched a learned model beat a hand-written rule.
2. **Rediscovered a real finding yourself** in ~10 lines of pandas.

No prior ML needed. Everything runs on the small anonymized dataset that ships with this repo — no credentials, no private data.

## 0. Setup (Kaggle, Colab, or local)
This Kaggle-compatible setup downloads **your** public FlyRank starter repository (including the sample CSV), sets the working directory to the repository root, and installs the starter requirements. **Enable Internet in Kaggle's notebook settings first.** The learning and discovery cells below are unchanged.


In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_KAGGLE = Path("/kaggle/working").is_dir()
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/rahadul231/flyrank-ml-internship-starter.git"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_KAGGLE or IN_COLAB:
    base_dir = Path("/kaggle/working" if IN_KAGGLE else "/content")
    repo_dir = base_dir / REPO_DIR
    data_file = repo_dir / "data/raw/content_refresh_anonymized.csv"

    if not data_file.is_file():
        if repo_dir.exists():
            raise RuntimeError(
                f"{repo_dir} exists but the starter CSV is missing. "
                "Restart this Kaggle/Colab session or remove the incomplete clone, then rerun."
            )
        print("Cloning your public FlyRank repository...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo_dir)], check=True)

    os.chdir(repo_dir)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # Local Python: find the existing repo root, if the notebook is run locally.
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Environment:", "Kaggle" if IN_KAGGLE else "Colab" if IN_COLAB else "local")
print("Working dir:", os.getcwd())
assert os.path.isfile("data/raw/content_refresh_anonymized.csv"), (
    "Starter CSV not found. If using Kaggle, enable Internet in Notebook Settings and rerun setup."
)
print("Starter data found. You're ready.")


## 1. Run the whole pipeline

This runs `scripts/run_all.py`: prepare features → baseline rule → train 3 models → evaluate → PDF.
It takes ~1 minute on the 30,000-row sample.

In [ ]:
# Watch it work — each of the 5 steps prints live as it runs (~1 minute total).
!{sys.executable} scripts/run_all.py


### What just happened?
The pipeline ranked every page for "refresh review" two ways: a **hand-written rule baseline** and a **learned model**. Let's compare them on **Precision@50** — of the top 50 pages each says to fix first, how many are actually declining?

In [ ]:
import json
res = json.load(open("outputs/model_results.json"))

base = res["baseline"]["baseline_precision_at_50"]
rf   = res["models"]["random_forest"]["precision_at_50"]

print(f"Hand-written rule  Precision@50: {base:.3f}   (~{round(base*50)} of the top 50 right)")
print(f"Random forest      Precision@50: {rf:.3f}   (~{round(rf*50)} of the top 50 right)")
print(f"\nThe learned model roughly {rf/base:.1f}x the rule on this metric.")
print("Validation split used:", res["split_strategy"], "(pages from a client are never in both train and test)")

You just ran a real ML system on real search data and saw a learned ranking beat a fixed rule. Now open `outputs/model_report.md` and skim it — that Markdown report is the *shape* of what your own capstone should produce.

## 2. Discover a real truth yourself

The safest, most satisfying early wins are **things you find in the data** — un-leakable, and they *are* the core lesson. Run the three cells below. Each is ~10 lines of pandas and each overturns a common SEO belief.

Every number is **computed live from the shipped CSV** — nothing is hardcoded.

In [ ]:
import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape[0], "rows,", df.shape[1], "columns")
df.head(3)

### Discovery A — "High search volume means more traffic." Does it?

In [ ]:
corr = df["search_volume"].corr(df["impressions_90d"])
print(f"Correlation between search_volume and impressions_90d: {corr:.3f}")
print("Near zero -> keyword search volume barely predicts the traffic a page actually gets.")

### Discovery B — the CTR cliff by position
Click-through rate is not flat: it collapses as you move down the results.

In [ ]:
visible = df[df["impressions_90d"] >= 100]
ctr_by_pos = visible.groupby("position_tier")["ctr"].mean().sort_values(ascending=False)
print(ctr_by_pos.round(4).to_string())
ctr_by_pos.plot(kind="bar", title="Mean CTR by position tier (impressions >= 100)", ylabel="CTR");

### Discovery C — is longer content the lever?
Compare word count for **declining** vs **growing** pages.

In [ ]:
wc = df.groupby("trend_direction")["word_count"].median()
print(wc.round(0).to_string())
print("\n'down' vs 'up' pages have almost the same median word count -> length is not the lever.")

## 3. 🔧 Your turn

Pick **one** of these and write a few lines below. This is your Week-1 discovery — you'll reference it in your Week-1 research-question write-up (on the InternHQ board).

- Redo Discovery A but only for pages with `impressions_90d > 0` — does the correlation change?
- In Discovery B, which `content_type` has the worst CTR *at the same position tier*?
- Find another belief to test: does `content_age_days` relate to `trend_direction`? Does `avg_position` relate to `ctr`?

**Rules:** describe what you observe as *observed / directional* — never "I proved Google's algorithm." Keep client data out of anything you publish.

In [ ]:

# Your Turn — Discovery A
# Does removing zero-impression pages change the correlation?

filtered = df[df["impressions_90d"] > 0]

original_corr = df["search_volume"].corr(df["impressions_90d"])
filtered_corr = filtered["search_volume"].corr(
    filtered["impressions_90d"]
)

print(f"Original correlation: {original_corr:.4f}")
print(f"Filtered correlation: {filtered_corr:.4f}")
print(f"Rows remaining: {len(filtered)}")


print("\nObservation:")
print(
    "Removing zero-impression pages did not change the"
    "correlation (0.0012 before and after filtering)."
    "All 30,000 rows remained, meaning none had zero impressions."
    "Keyword search volume has almost no linear relationship"
    "with search impressions in this dataset."
)


### Save your work
**Colab:** *File → Save a copy in GitHub* (writes to your own repo — that's your submission) and *File → Save a copy in Drive* (so the session doesn't evaporate).

Next: `02_your_first_readable_model.ipynb` — where the model becomes a rule you can read.